# CS781 Phase 1 — RWKV training and inference (cleaned)

This version removes the conflicting experimental/debug cells and keeps one model definition, one resumable training loop, and the existing inference/export cells. Run top to bottom. GPU is strongly recommended; the training cell will stop with a clear message if no CUDA GPU is available.

Checkpointing: local atomic checkpoint every 100 optimizer steps; optional private Kaggle Dataset upload every 250 steps. To resume after a session reset, attach the checkpoint dataset to the notebook inputs.


In [ ]:
import os
import gc
import json
import math
import time
import random
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")
TRAIN_MANIFEST = DATA_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = DATA_ROOT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = DATA_ROOT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = DATA_ROOT / "tokenizer/legal_bpe_32k.json"

WORK_DIR = Path("/kaggle/working/phase1_rwkv")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from tokenizers import Tokenizer, decoders

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel() # Required to prevent Ġ / Ċ artifacts

VOCAB_SIZE = tokenizer.get_vocab_size()
print("Vocabulary:", VOCAB_SIZE)
assert VOCAB_SIZE == 32000

In [ ]:
# Reuse the existing WorkerSafeDocumentAwarePacker or LegalDataset from previous experiments
# Ensure standard causal-language-model labels (shifted).
CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

def load_manifest(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_manifest(TRAIN_MANIFEST)

class LegalDataset(torch.utils.data.Dataset):
    def __init__(self, records, tokenizer, context_length):
        self.records = records
        self.tokenizer = tokenizer
        self.context_length = context_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        text = self.records[idx].get("text", "")
        ids = self.tokenizer.encode(text).ids
        if len(ids) >= self.context_length + 1:
            ids = ids[:self.context_length + 1]
        else:
            ids = ids + [0] * ((self.context_length + 1) - len(ids))
        
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return {"input_ids": x, "labels": y}

train_dataset = LegalDataset(train_records, tokenizer, CONTEXT_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
print("DataLoader Ready. Context Length:", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# CELL 8 — RWKV ARCHITECTURE WITH FP32 HEAD BOUNDARY
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

@torch.jit.script
def wkv_forward_jit(
    k: torch.Tensor, v: torch.Tensor, r: torch.Tensor, 
    time_decay: torch.Tensor, time_first: torch.Tensor, 
    num: torch.Tensor, den: torch.Tensor, p: torch.Tensor
):
    B: int = k.size(0)
    T: int = k.size(1)
    
    out = torch.empty_like(v)
    w = -torch.exp(time_decay)
    u = time_first
    
    for t in range(T):
        kt = k[:, t, :]
        vt = v[:, t, :]
        rt = r[:, t, :]
        
        p_curr = torch.maximum(p, kt + u)
        e1 = torch.exp(p - p_curr)
        e2 = torch.exp(kt + u - p_curr)
        
        den_out = torch.clamp(e1 * den + e2, min=1e-8)
        wkv = (e1 * num + e2 * vt) / den_out
        
        out[:, t, :] = torch.sigmoid(rt) * wkv
        
        p_next = torch.maximum(p + w, kt)
        e1_next = torch.exp(p + w - p_next)
        e2_next = torch.exp(kt - p_next)
        
        num = e1_next * num + e2_next * vt
        den = e1_next * den + e2_next
        p = p_next
        
    return out, num, den, p

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        self.time_decay = nn.Parameter(torch.full((d_model,), -2.0))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x).float()
        
        rwkv, num, den, p = wkv_forward_jit(
            k, v, r, 
            self.time_decay.float(), 
            self.time_first.float(), 
            num, den, p
        )
        
        out = self.output(rwkv.to(x.dtype))
        return out, (x[:, -1:, :], num, den, p)

class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        hidden_size = 4 * d_model
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype) if state is None else state[0]
            
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x)
        k = torch.relu(k.float()) ** 2
        k = torch.clamp(k, max=65500.0) 
        
        v = self.value(k.to(x.dtype))
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        return out, (x[:, -1:, :], )

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        return x, (new_tm_state, new_cm_state)

class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        x = self.ln_out(x)

        # Inference needs logits; training computes cross-entropy in chunks
        # to avoid materializing [batch, sequence, vocabulary] all at once.
        if labels is None:
            with torch.autocast(device_type=x.device.type, enabled=False):
                return self.head(x.float()), None

        x_flat = x[:, :-1, :].contiguous().view(-1, x.size(-1))
        y_flat = labels[:, 1:].contiguous().view(-1)
        chunk_size = 256
        total_loss = torch.zeros((), device=x.device, dtype=torch.float32)
        total_tokens = 0

        with torch.autocast(device_type=x.device.type, enabled=False):
            for offset in range(0, x_flat.size(0), chunk_size):
                end = min(offset + chunk_size, x_flat.size(0))
                logits_chunk = self.head(x_flat[offset:end].float())
                target_chunk = y_flat[offset:end]
                total_loss = total_loss + F.cross_entropy(
                    logits_chunk, target_chunk, reduction="sum"
                )
                total_tokens += target_chunk.numel()

        return None, total_loss / max(total_tokens, 1)

model = RWKVLanguageModel().to(DEVICE)
model.head.float() # Keep final projection in FP32
print(f"✓ Model Initialized with FP32 Head ({sum(p.numel() for p in model.parameters()):,} parameters)")

In [ ]:
# CS781 RWKV — PRE-TRAINING SMOKE TEST
import torch

print("=" * 55)
print("CS781 RWKV PRE-TRAINING TEST")
print("=" * 55)

assert "model" in globals(), "Model not initialized. Run the model setup cell."
assert "train_loader" in globals(), "Training data not initialized."
assert torch.cuda.is_available(), (
    "GPU unavailable. Enable GPU in Kaggle Settings and restart."
)

device = next(model.parameters()).device
assert device.type == "cuda", f"Model is on {device}, not GPU."

model.train()
model.zero_grad(set_to_none=True)

# Load one batch and shorten the sequence to reduce test memory.
batch = next(iter(train_loader))
test_len = min(128, batch["input_ids"].shape[1])

input_ids = batch["input_ids"][:, :test_len].to(device)
labels = batch["labels"][:, :test_len].to(device)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Input shape: {tuple(input_ids.shape)}")
print("Running forward pass...")

try:
    with torch.autocast(device_type="cuda", enabled=False):
        logits, loss = model(input_ids, labels=labels)

    assert loss is not None, "Model did not return a training loss."
    assert torch.isfinite(loss).item(), f"Non-finite loss: {loss.item()}"

    print(f"Forward pass OK | Loss: {loss.item():.4f}")
    print("Running backward pass...")

    loss.backward()

    grads = [
        p.grad for p in model.parameters()
        if p.requires_grad and p.grad is not None
    ]
    assert grads, "No gradients were produced."
    assert all(torch.isfinite(g).all().item() for g in grads), (
        "NaN or Inf detected in gradients."
    )

    print(f"Backward pass OK | Gradients: {len(grads)} tensors")
    print("PASS: model, data, loss, and gradients are working.")
    print("No optimizer step performed; model weights were not updated.")

except Exception as e:
    print(f"TEST FAILED: {type(e).__name__}: {e}")
    raise

finally:
    model.zero_grad(set_to_none=True)
    del batch, input_ids, labels
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 55)
print("If PASS, proceed to the training cell.")
print("=" * 55)

In [ ]:
# ============================================================
# RWKV TRAINING — RESUMABLE + OPTIONAL KAGGLE DATASET CHECKPOINTS
# ============================================================
import os, time, gc, json, shutil, subprocess, random
from pathlib import Path
import torch

# Avoid silently attempting a very slow 137M-parameter CPU training run.
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached. In Kaggle, open Notebook Settings → Accelerator → "
        "GPU (T4×2 if available), save, restart the session, then run the notebook "
        "from the first cell. Installing CUDA PyTorch alone cannot attach a GPU."
    )

MAX_STEPS = 1000
GRAD_ACCUM = int(GRAD_ACCUM_STEPS)
MAX_GRAD_NORM = 1.0
LOCAL_SAVE_EVERY = 100
KAGGLE_UPLOAD_EVERY = 250
LR = 5e-5
WEIGHT_DECAY = 0.01

WORK_DIR = Path("/kaggle/working/phase1_rwkv")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WORK_DIR / "checkpoint_latest.pt"
LOG_PATH = WORK_DIR / "training_log.jsonl"

CHECKPOINT_DATASET_SLUG = "oipranesh/cs781-phase1-rwkv-checkpoints"
UPLOAD_DIR = Path("/kaggle/working/rwkv_checkpoint_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Credentials are optional; local checkpointing does not depend on them.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    print("Kaggle API credentials loaded.")
except Exception as exc:
    print("Kaggle Dataset uploads unavailable; local checkpointing remains enabled.")
    print("Reason:", str(exc)[:250])

model = model.to(DEVICE).float()
model.head.float()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def find_input_checkpoint():
    candidates = list(Path("/kaggle/input").rglob("checkpoint_latest.pt"))
    valid = []
    for path in candidates:
        try:
            obj = torch.load(path, map_location="cpu", weights_only=False)
            if isinstance(obj, dict) and "model_state_dict" in obj and "step" in obj:
                valid.append((int(obj["step"]), path))
        except Exception as exc:
            print("Skipping unreadable checkpoint:", path, str(exc)[:160])
    return max(valid, default=(0, None), key=lambda pair: pair[0])[1]

def save_checkpoint(step, reason="periodic"):
    checkpoint = {
        "step": int(step),
        "optimizer_step": int(step),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "reason": reason,
        "saved_unix_time": time.time(),
        "model_config": {
            "architecture": "RWKVLanguageModel",
            "vocab_size": 32000,
            "d_model": 768,
            "num_layers": 12,
            "context_length": CONTEXT_LENGTH,
            "grad_accum": GRAD_ACCUM,
        },
        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),
    }
    tmp_path = Path(str(CHECKPOINT_PATH) + ".tmp")
    torch.save(checkpoint, tmp_path)
    os.replace(tmp_path, CHECKPOINT_PATH)
    (WORK_DIR / "training_state.json").write_text(json.dumps({
        "step": int(step), "reason": reason,
        "saved_unix_time": time.time(),
        "checkpoint_file": CHECKPOINT_PATH.name,
    }, indent=2))
    print(f"✓ Saved checkpoint at step {step}: {CHECKPOINT_PATH} "
          f"({CHECKPOINT_PATH.stat().st_size / 1024**3:.2f} GiB)")

def upload_checkpoint_to_kaggle(step):
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Upload skipped: configure KAGGLE_USERNAME and KAGGLE_KEY in Kaggle Secrets.")
        return False

    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    (UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps({
        "title": "CS781 Phase 1 RWKV Checkpoints",
        "id": CHECKPOINT_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
        "private": True
    }, indent=2))
    shutil.copy2(CHECKPOINT_PATH, UPLOAD_DIR / "checkpoint_latest.pt")
    state_file = WORK_DIR / "training_state.json"
    if state_file.exists():
        shutil.copy2(state_file, UPLOAD_DIR / "training_state.json")
    if LOG_PATH.exists():
        shutil.copy2(LOG_PATH, UPLOAD_DIR / "training_log.jsonl")

    for command in (
        ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
         "-m", f"RWKV checkpoint step {step}", "--dir-mode", "tar"],
        ["kaggle", "datasets", "create", "-p", str(UPLOAD_DIR), "--dir-mode", "tar"],
    ):
        try:
            result = subprocess.run(command, text=True, capture_output=True, timeout=1800)
            if result.returncode == 0:
                print("✓ Kaggle checkpoint Dataset uploaded/updated.")
                if result.stdout:
                    print(result.stdout[-1000:])
                return True
            print((result.stderr or result.stdout)[-1200:])
        except Exception as exc:
            print("Kaggle upload error:", repr(exc))
    print("Upload failed; local checkpoint is still available.")
    return False

start_step = 0
resume_path = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else find_input_checkpoint()
if resume_path is not None:
    print("Loading checkpoint:", resume_path)
    ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    if ckpt.get("optimizer_state_dict") is not None:
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as exc:
            print("Optimizer state could not be restored; continuing with a fresh optimizer:", repr(exc))
    start_step = int(ckpt.get("step", 0))
    try:
        if ckpt.get("python_rng_state") is not None:
            random.setstate(ckpt["python_rng_state"])
        if ckpt.get("torch_rng_state") is not None:
            torch.set_rng_state(ckpt["torch_rng_state"].cpu())
        if ckpt.get("cuda_rng_state_all") is not None:
            torch.cuda.set_rng_state_all(ckpt["cuda_rng_state_all"])
    except Exception as exc:
        print("RNG restore warning:", repr(exc))
    print(f"✓ Resuming from step {start_step}")
else:
    print("No checkpoint found; starting at step 0.")

if start_step >= MAX_STEPS:
    print(f"Checkpoint already reached step {start_step}; MAX_STEPS={MAX_STEPS}. No training needed.")
else:
    model.train()
    train_iter = iter(train_loader)

    def get_batch():
        global train_iter
        try:
            return next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            return next(train_iter)

    training_start = time.time()
    for step in range(start_step + 1, MAX_STEPS + 1):
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0
        step_start = time.time()

        for _ in range(GRAD_ACCUM):
            batch = get_batch()
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            _, loss = model(input_ids=input_ids, labels=labels)
            if loss is None or not torch.isfinite(loss).item():
                raise RuntimeError(f"Invalid loss at optimizer step {step}: {loss}")
            loss_sum += loss.detach().item()
            (loss / GRAD_ACCUM).backward()
            del input_ids, labels, batch, loss

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        if not torch.isfinite(torch.as_tensor(grad_norm)).item():
            raise RuntimeError(f"Non-finite gradient norm at optimizer step {step}")
        optimizer.step()

        avg_loss = loss_sum / GRAD_ACCUM
        step_seconds = time.time() - step_start
        if step == start_step + 1 or step % 10 == 0 or step == MAX_STEPS:
            record = {
                "step": step, "loss": avg_loss,
                "grad_norm": float(torch.as_tensor(grad_norm).item()),
                "step_seconds": step_seconds,
                "elapsed_seconds": time.time() - training_start,
            }
            with LOG_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps(record) + "\n")
            print(f"step={step}/{MAX_STEPS} | loss={avg_loss:.4f} | "
                  f"grad={record['grad_norm']:.3f} | {step_seconds:.1f}s/step")

        if step % LOCAL_SAVE_EVERY == 0 or step == MAX_STEPS:
            save_checkpoint(step, "final" if step == MAX_STEPS else "periodic")
        if step % KAGGLE_UPLOAD_EVERY == 0 or step == MAX_STEPS:
            upload_checkpoint_to_kaggle(step)
        if step % 25 == 0:
            gc.collect()

print("Training cell finished.")
print("Latest local checkpoint:", CHECKPOINT_PATH)
print("Training log:", LOG_PATH)


In [ ]:
# ============================================================
# RWKV FINAL INFERENCE — LOCATE OFFICIAL TEST FILE
# ============================================================

from pathlib import Path
import json

candidates = list(Path("/kaggle/input").rglob("test.json"))

print("Found:")
for p in candidates:
    print(" ", p)

if not candidates:
    raise FileNotFoundError(
        "Could not find Phase1-Test.json under /kaggle/input"
    )

TEST_JSON = candidates[0]

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print()
print("Test file:", TEST_JSON)
print("Number of examples:", len(test_data))

first_id = next(iter(test_data))
print("First ID:", first_id)
print("Prompt preview:")
print(test_data[first_id]["text"][:300])

In [ ]:
# ============================================================
# LOAD FINAL RWKV CHECKPOINT
# ============================================================

from pathlib import Path
import torch

CHECKPOINT_PATH = Path(
    "/kaggle/working/phase1_rwkv/checkpoint_latest.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint step:", checkpoint.get("step"))
print("Checkpoint keys:", list(checkpoint.keys()))

# Your ACTUAL RWKV constructor
model = RWKVLanguageModel(
    vocab_size=32000,
    d_model=768,
    num_layers=12
).to(DEVICE).float()

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print()
print("✓ Final RWKV checkpoint loaded")
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)
print("Device:", DEVICE)

In [ ]:
# ============================================================
# TOKENIZER
# ============================================================

from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/"
    "tokenizer/legal_bpe_32k.json"
)

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found: {TOKENIZER_PATH}"
    )

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

# Make decoded text human-readable
tokenizer.decoder = decoders.ByteLevel()

print("✓ Tokenizer loaded")
print("Vocab size:", tokenizer.get_vocab_size())

In [ ]:
# ============================================================
# RWKV GENERATION — CORRECTED FOR YOUR MODEL
# ============================================================

import torch

def normalize_text(text):
    return " ".join(
        text.replace("Ġ", " ")
            .replace("Ċ", " ")
            .split()
    )


def generate_128_words(
    prompt,
    max_new_tokens=300,
    temperature=0.8,
    top_k=50,
):
    encoded = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
        device=DEVICE
    )

    prompt_len = input_ids.shape[1]

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Keep within the model's 4096-token context
            context_ids = input_ids[:, -4096:]

            # YOUR RWKV MODEL RETURNS:
            # logits, loss
            logits, _ = model(
                input_ids=context_ids
            )

            # Last-token prediction
            logits = logits[:, -1, :]

            # Temperature
            logits = logits / temperature

            # Top-k sampling
            if top_k is not None and top_k > 0:
                k = min(top_k, logits.shape[-1])

                values, indices = torch.topk(
                    logits,
                    k
                )

                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered_logits.scatter_(
                    1,
                    indices,
                    values
                )

                logits = filtered_logits

            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

            # Decode ONLY newly generated tokens
            generated_ids = input_ids[
                0,
                prompt_len:
            ].tolist()

            generated_text = normalize_text(
                tokenizer.decode(generated_ids)
            )

            words = generated_text.split()

            if len(words) >= 128:
                return " ".join(words[:128])

    # Fallback if 128 words weren't reached
    generated_ids = input_ids[
        0,
        prompt_len:
    ].tolist()

    generated_text = normalize_text(
        tokenizer.decode(generated_ids)
    )

    words = generated_text.split()

    return " ".join(words[:128])

In [ ]:
# ============================================================
# RWKV GENERATION DIAGNOSTIC
# ============================================================

import torch

test_id = next(iter(test_data))
prompt = test_data[test_id]["text"]

encoded = tokenizer.encode(prompt)

print("Prompt characters :", len(prompt))
print("Prompt tokens     :", len(encoded.ids))
print("First 20 token IDs :", encoded.ids[:20])
print("Last 20 token IDs  :", encoded.ids[-20:])

# Check tokenizer decoding
print("\nTokenizer test:")
print("Decoded prompt chars:", len(tokenizer.decode(encoded.ids)))
print("Decoded preview:", repr(tokenizer.decode(encoded.ids[:50])))

# Run ONE model prediction
input_ids = torch.tensor(
    [encoded.ids[-4096:]],
    dtype=torch.long,
    device=DEVICE
)

model.eval()

with torch.no_grad():
    logits, loss = model(input_ids=input_ids)

print("\nModel output:")
print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())

# Inspect top predicted tokens
last_logits = logits[:, -1, :]
values, indices = torch.topk(last_logits, 20, dim=-1)

print("\nTop 20 predicted tokens:")
for rank, (value, idx) in enumerate(
    zip(values[0].tolist(), indices[0].tolist()),
    start=1
):
    decoded = tokenizer.decode([idx])
    print(
        f"{rank:2d}. "
        f"ID={idx:6d} "
        f"logit={value:10.4f} "
        f"decoded={repr(decoded)}"
    )

In [ ]:
# ============================================================
# RWKV FINAL GENERATION
# Prevent PAD token from being generated
# ============================================================

def generate_128_words(
    prompt,
    max_new_tokens=600,
    temperature=0.8,
    top_k=50,
):
    encoded = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
        device=DEVICE
    )

    prompt_len = input_ids.shape[1]

    PAD_TOKEN_ID = 0

    with torch.no_grad():

        for step in range(max_new_tokens):

            context_ids = input_ids[:, -4096:]

            logits, _ = model(
                input_ids=context_ids
            )

            logits = logits[:, -1, :]

            # NEVER generate padding token
            logits[:, PAD_TOKEN_ID] = float("-inf")

            # Temperature
            logits = logits / temperature

            # Top-k
            if top_k is not None and top_k > 0:

                k = min(top_k, logits.shape[-1])

                values, indices = torch.topk(
                    logits,
                    k
                )

                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered_logits.scatter_(
                    1,
                    indices,
                    values
                )

                logits = filtered_logits

            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

            # Decode generated portion
            generated_ids = input_ids[
                0,
                prompt_len:
            ].tolist()

            generated_text = normalize_text(
                tokenizer.decode(generated_ids)
            )

            words = generated_text.split()

            if len(words) >= 128:
                return " ".join(words[:128])

    # Don't silently produce an invalid submission
    generated_ids = input_ids[
        0,
        prompt_len:
    ].tolist()

    generated_text = normalize_text(
        tokenizer.decode(generated_ids)
    )

    words = generated_text.split()

    raise RuntimeError(
        f"Reached {max_new_tokens} generated tokens "
        f"but only produced {len(words)} words."
    )

In [ ]:
# ============================================================
# TEST FINAL GENERATION
# ============================================================

test_id = next(iter(test_data))
prompt = test_data[test_id]["text"]

prediction = generate_128_words(
    prompt,
    max_new_tokens=600,
    temperature=0.8,
    top_k=50,
)

print("Generated words:", len(prediction.split()))
print()
print("=" * 70)
print(prediction)
print("=" * 70)

assert len(prediction.split()) == 128

print()
print("✓ EXACTLY 128 WORDS")

In [ ]:
# ============================================================
# GENERATE FINAL RWKV prediction.json
# ============================================================

import json
import time
from pathlib import Path

OUTPUT_JSON = Path("/kaggle/working/prediction.json")

predictions = {}
total = len(test_data)

start_time = time.time()

for i, (example_id, example) in enumerate(test_data.items(), start=1):

    prompt = example["text"]

    continuation = generate_128_words(
        prompt,
        max_new_tokens=600,
        temperature=0.8,
        top_k=50,
    )

    word_count = len(continuation.split())

    if word_count != 128:
        raise RuntimeError(
            f"ID {example_id}: generated "
            f"{word_count} words instead of 128"
        )

    predictions[str(example_id)] = {
        "text": continuation
    }

    print(
        f"[{i}/{total}] "
        f"ID={example_id} | "
        f"words={word_count}"
    )

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

elapsed = time.time() - start_time

print()
print("=" * 70)
print("PREDICTION GENERATION COMPLETE")
print("=" * 70)
print("File:", OUTPUT_JSON)
print("Examples:", len(predictions))
print(f"Time: {elapsed / 60:.2f} minutes")

In [ ]:
# ============================================================
# VALIDATE FINAL prediction.json
# ============================================================

with open(TEST_JSON, "r", encoding="utf-8") as f:
    official_test = json.load(f)

with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    prediction = json.load(f)

expected_ids = {str(x) for x in official_test.keys()}
actual_ids = {str(x) for x in prediction.keys()}

assert expected_ids == actual_ids, (
    f"ID mismatch. Missing: "
    f"{expected_ids - actual_ids}, "
    f"Extra: {actual_ids - expected_ids}"
)

for example_id, item in prediction.items():

    assert isinstance(item, dict)
    assert "text" in item
    assert isinstance(item["text"], str)

    word_count = len(item["text"].split())

    assert word_count == 128, (
        f"ID {example_id}: {word_count} words"
    )

print("✓ IDs exactly match official test")
print("✓ No missing/extra IDs")
print("✓ All predictions contain text")
print("✓ All predictions contain exactly 128 words")
print("✓ prediction.json is structurally valid")

In [ ]:
# ============================================================
# CREATE SUBMISSION ZIP
# ============================================================

import zipfile

ZIP_PATH = Path(
    "/kaggle/working/phase1_rwkv_submission.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(
        OUTPUT_JSON,
        arcname="prediction.json"
    )

print("✓ Created:", ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("ZIP contents:", z.namelist())

assert z.namelist() == ["prediction.json"]

print("✓ prediction.json is at ZIP root")